In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.io import savemat
from scipy.io import loadmat

In [ ]:
project_path = '../surrogate-training/'
import sys
sys.path.append(project_path)
from Unet import create_vae
# Load model
vae_model = create_vae()
vae_model.load_weights(project_path +'best_model_mae.weights.h5')

In [ ]:
# load training data
perm_CSS = loadmat(project_path + 'training_data.mat')['perm_CSS']
perm_CSC = loadmat(project_path +'training_data.mat')['perm_CSC']
perm_SCC = loadmat(project_path +'training_data.mat')['perm_SCC']
perm_SCS = loadmat(project_path +'training_data.mat')['perm_SCS']
perm_train = np.concatenate((perm_CSS,perm_CSC,perm_SCC,perm_SCS),axis = -1)
perm_train = np.reshape(np.log10(perm_train), (100, 100, 2, -1))
perm_train = np.flip(perm_train, axis=0)
perm_train = np.transpose(perm_train, [3, 0, 1, 2])
print('train input shape:',perm_train.shape)

Perm_CSS = loadmat(project_path +'training_data.mat')['Perm_CSS']
Perm_CSC = loadmat(project_path +'training_data.mat')['Perm_CSC']
Perm_SCC = loadmat(project_path +'training_data.mat')['Perm_SCC']
Perm_SCS = loadmat(project_path +'training_data.mat')['Perm_SCS']
Perm_train = np.concatenate((Perm_CSS,Perm_CSC,Perm_SCC,Perm_SCS),axis = -1)
Perm_train = np.log10(Perm_train[[0,2],:].T)
print('train output shape:',Perm_train.shape)

# load testing data
perm_CSS = loadmat(project_path +'validation_data.mat')['perm_CSS']
perm_CSC = loadmat(project_path +'validation_data.mat')['perm_CSC']
perm_SCC = loadmat(project_path +'validation_data.mat')['perm_SCC']
perm_SCS = loadmat(project_path +'validation_data.mat')['perm_SCS']
perm_test = np.concatenate((perm_CSS,perm_CSC,perm_SCC,perm_SCS),axis = -1)
perm_test = np.reshape(np.log10(perm_test), (100, 100, 2, -1))
perm_test = np.flip(perm_test, axis=0)
perm_test = np.transpose(perm_test, [3, 0, 1, 2])
print('test input shape:',perm_test.shape)

Perm_CSS = loadmat(project_path +'validation_data.mat')['Perm_CSS']
Perm_CSC = loadmat(project_path +'validation_data.mat')['Perm_CSC']
Perm_SCC = loadmat(project_path +'validation_data.mat')['Perm_SCC']
Perm_SCS = loadmat(project_path +'validation_data.mat')['Perm_SCS']
Perm_test = np.concatenate((Perm_CSS,Perm_CSC,Perm_SCC,Perm_SCS),axis = -1)
Perm_test = np.log10(Perm_test[[0,2],:].T)
print('test output shape:',Perm_test.shape)

In [ ]:
def min_max_scale_X(X):
    X_min = np.min(X,axis = (0,1,2))
    X_max = np.max(X,axis = (0,1,2))
    X_scaled = (X-X_min)/(X_max-X_min)-0.5
    return X_scaled, X_min,X_max

def min_max_scale_back_X(X_scaled,X_min,X_max):
    X = (X_scaled+0.5)*(X_max-X_min)+X_min
    return X

def min_max_scale_Y(Y):
    Y_min = np.min(Y,axis = 0)
    Y_max = np.max(Y,axis = 0)
    Y_scaled = (Y-Y_min)/(Y_max-Y_min)-0.5
    return Y_scaled, Y_min,Y_max

def min_max_scale_back_Y(Y_scaled,Y_min,Y_max):
    Y = (Y_scaled+0.5)*(Y_max-Y_min)+Y_min
    return Y

def padding_128(X):
  X_pad = np.zeros((X.shape[0],128,128,X.shape[3]))
  for i in range(X.shape[0]):
    X_pad[i,13:13+X.shape[1],13:13+X.shape[2],:] = X[i,:,:,:]
  return X_pad

Y_train,Y_min,Y_max = min_max_scale_Y(Perm_train)
X_train,X_min,X_max = min_max_scale_X(perm_train)

In [ ]:
# Predict a representative scenario
CASE = 'CSS'
print('Predict Scenario %s\n' % CASE)

perm_test = loadmat(project_path +'validation_data.mat')[f'perm_{CASE}']
perm_test = np.reshape(np.log10(perm_test), (100, 100, 2, -1))
perm_test = np.flip(perm_test, axis=0)
perm_test = np.transpose(perm_test, [3, 0, 1, 2])
print('test input shape:',perm_test.shape)

Perm_test = loadmat(project_path +'validation_data.mat')[f'Perm_{CASE}']
Perm_test = np.log10(Perm_test[[0,2],:].T)
print('test output shape:',Perm_test.shape)

Y_test = (Perm_test-Y_min)/(Y_max-Y_min)-0.5
X_test = (perm_test-X_min)/(X_max-X_min)-0.5
X_test = padding_128(X_test)
Y_pred = vae_model.predict(X_test)
Perm_pred = min_max_scale_back_Y(Y_pred,Y_min,Y_max)

savemat(project_path +f'val_{CASE}.mat', {'Perm_pred': Perm_pred, 'Perm_test': Perm_test})

# Compute 1D Wasserstein distance
from scipy.stats import wasserstein_distance
distance_x = wasserstein_distance(Perm_pred[:,0], Perm_test[:,0])
print("Wasserstein distance Kxx:", distance_x)
distance_z = wasserstein_distance(Perm_pred[:,1], Perm_test[:,1])
print("Wasserstein distance Kzz:", distance_z)

# sample-wise visualization
plt.figure(figsize=(10,6))
plt.subplot(2,2,1)
plt.plot(Perm_pred[:,0],np.arange(1,1001),'d')
plt.plot(Perm_test[:,0],np.arange(1,1001),'o')
plt.xlabel('log(Kxx)')
plt.legend(['pred','true'], loc='upper left')

plt.subplot(2,2,2)
plt.plot(Perm_pred[:,1],np.arange(1,1001),'d')
plt.plot(Perm_test[:,1],np.arange(1,1001),'o')
plt.xlabel('log(Kzz)')
plt.legend(['pred','true'], loc='upper left')

plt.subplot(2,2,3)
plt.hist(Perm_pred[:,0],alpha = 0.5,density = True)
plt.hist(Perm_test[:,0],alpha = 0.5,density = True)
plt.xlabel('log(Kxx)')
plt.legend(['test pred','test true','train true'], loc='upper left')

plt.subplot(2,2,4)
plt.hist(Perm_pred[:,1],alpha = 0.5,density = True)
plt.hist(Perm_test[:,1],alpha = 0.5,density = True)
plt.xlabel('log(Kzz)')
plt.legend(['test pred','test true','train true'], loc='upper left')

plt.tight_layout()
#plt.savefig(f'Perm_test_{CASE}.png', bbox_inches='tight')
plt.show()